In [ ]:
"""
=============================================================================
SAM (SEGMENT ANYTHING MODEL) PAPER DETECTION PIPELINE - COMPLETE VERSION
=============================================================================
Enhanced with geometric filtering to reliably detect papers even when
background is larger than the paper itself.

INSTALLATION INSTRUCTIONS:
--------------------------
Run these commands in your terminal with UV:

    uv pip install segment-anything
    uv pip install git+https://github.com/facebookresearch/segment-anything.git

Or just:
    uv pip install segment-anything

=============================================================================
"""

import cv2
import numpy as np
import matplotlib.pyplot as plt
import torch
from segment_anything import sam_model_registry, SamAutomaticMaskGenerator, SamPredictor
import urllib.request
from pathlib import Path
from typing import List, Dict, Tuple, Optional


# =============================================================================
# GEOMETRIC FILTERING FUNCTIONS
# =============================================================================

def is_paper_like(mask: np.ndarray, img_shape: Tuple[int, int],
                  min_area_ratio: float = 0.05,
                  max_area_ratio: float = 0.85,
                  min_aspect_ratio: float = 0.5,
                  max_aspect_ratio: float = 2.5,
                  min_rectangularity: float = 0.7) -> Tuple[bool, Dict]:
    """
    Check if a mask represents a paper-like object based on geometric properties

    Args:
        mask: Binary mask (0-255)
        img_shape: (height, width) of original image
        min_area_ratio: Minimum area as fraction of image (exclude tiny objects)
        max_area_ratio: Maximum area as fraction of image (exclude background)
        min_aspect_ratio: Minimum width/height ratio
        max_aspect_ratio: Maximum width/height ratio
        min_rectangularity: How rectangular it should be (0-1)

    Returns:
        (is_paper, metrics) where metrics contains diagnostic info
    """

    img_area = img_shape[0] * img_shape[1]

    # Find contours
    contours, _ = cv2.findContours(mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)

    if not contours:
        return False, {"reason": "no_contours"}

    # Get largest contour
    contour = max(contours, key=cv2.contourArea)
    area = cv2.contourArea(contour)

    # Metric 1: Area ratio (not too small, not too large)
    area_ratio = area / img_area

    if area_ratio < min_area_ratio:
        return False, {"reason": "too_small", "area_ratio": area_ratio}

    if area_ratio > max_area_ratio:
        return False, {"reason": "too_large_likely_background", "area_ratio": area_ratio}

    # Metric 2: Aspect ratio (roughly rectangular)
    rect = cv2.minAreaRect(contour)
    width, height = rect[1]

    if width == 0 or height == 0:
        return False, {"reason": "degenerate_shape"}

    aspect_ratio = max(width, height) / min(width, height)

    if aspect_ratio < min_aspect_ratio or aspect_ratio > max_aspect_ratio:
        return False, {"reason": "bad_aspect_ratio", "aspect_ratio": aspect_ratio}

    # Metric 3: Rectangularity (how well it fills its bounding box)
    bbox_area = width * height
    rectangularity = area / bbox_area if bbox_area > 0 else 0

    if rectangularity < min_rectangularity:
        return False, {"reason": "not_rectangular", "rectangularity": rectangularity}

    # Metric 4: Number of corners (should have ~4 for paper)
    epsilon = 0.02 * cv2.arcLength(contour, True)
    approx = cv2.approxPolyDP(contour, epsilon, True)
    num_corners = len(approx)

    # Papers typically have 4 corners, allow 4-8 (might be slightly curved)
    if num_corners < 4 or num_corners > 8:
        return False, {"reason": "wrong_corner_count", "num_corners": num_corners}

    # Metric 5: Convexity (paper should be mostly convex)
    hull = cv2.convexHull(contour)
    hull_area = cv2.contourArea(hull)
    solidity = area / hull_area if hull_area > 0 else 0

    if solidity < 0.8:  # Paper should be at least 80% convex
        return False, {"reason": "not_convex", "solidity": solidity}

    # ALL CHECKS PASSED - This is paper-like!
    metrics = {
        "reason": "PAPER_DETECTED",
        "area_ratio": area_ratio,
        "aspect_ratio": aspect_ratio,
        "rectangularity": rectangularity,
        "num_corners": num_corners,
        "solidity": solidity,
        "area_pixels": int(area),
        "score": (rectangularity + solidity) / 2  # Combined quality score
    }

    return True, metrics


def find_best_paper_mask(all_masks: List[Dict], img_shape: Tuple[int, int],
                         verbose: bool = True) -> Tuple[Optional[np.ndarray], Dict]:
    """
    Find the best paper candidate from all SAM masks using geometric filtering

    Args:
        all_masks: List of masks from SAM's automatic segmentation
        img_shape: (height, width) of original image
        verbose: Print diagnostic info

    Returns:
        (best_mask, best_metrics) or (None, {}) if no paper found
    """

    if verbose:
        print(f"\n🔍 Analyzing {len(all_masks)} detected objects...")
        print(f"   Image size: {img_shape[1]}×{img_shape[0]} pixels")

    paper_candidates = []

    for i, mask_dict in enumerate(all_masks):
        mask = (mask_dict['segmentation'].astype(np.uint8)) * 255

        is_paper, metrics = is_paper_like(mask, img_shape)

        if is_paper:
            paper_candidates.append({
                'index': i,
                'mask': mask,
                'metrics': metrics,
                'sam_area': mask_dict['area']
            })

            if verbose:
                print(f"   ✓ Object {i+1}: PAPER CANDIDATE")
                print(f"      Area ratio: {metrics['area_ratio']:.1%}")
                print(f"      Aspect ratio: {metrics['aspect_ratio']:.2f}")
                print(f"      Rectangularity: {metrics['rectangularity']:.2%}")
                print(f"      Corners: {metrics['num_corners']}")
                print(f"      Solidity: {metrics['solidity']:.2%}")
                print(f"      Score: {metrics['score']:.3f}")
        else:
            if verbose:
                reason = metrics.get('reason', 'unknown')
                detail = ""
                if 'area_ratio' in metrics:
                    detail = f" (area: {metrics['area_ratio']:.1%})"
                elif 'aspect_ratio' in metrics:
                    detail = f" (aspect: {metrics['aspect_ratio']:.2f})"
                print(f"   ✗ Object {i+1}: Rejected - {reason}{detail}")

    if not paper_candidates:
        if verbose:
            print("\n❌ No paper-like objects found!")
            print("   Try adjusting detection thresholds or check image quality")
        return None, {}

    # Sort by quality score
    paper_candidates.sort(key=lambda x: x['metrics']['score'], reverse=True)

    best = paper_candidates[0]

    if verbose:
        print(f"\n✅ Selected best paper candidate: Object {best['index']+1}")
        print(f"   Quality score: {best['metrics']['score']:.3f}")
        if len(paper_candidates) > 1:
            print(f"   (Found {len(paper_candidates)} paper candidates total)")

    return best['mask'], best['metrics']


def visualize_paper_detection_analysis(img: np.ndarray, all_masks: List[Dict],
                                       max_display: int = 6):
    """
    Show detailed analysis of why objects were classified as paper or not

    Args:
        img: Original image
        all_masks: All masks from SAM
        max_display: Maximum number of objects to display
    """
    n = min(len(all_masks), max_display)
    fig, axes = plt.subplots(2, n, figsize=(4*n, 8))

    if n == 1:
        axes = axes.reshape(2, 1)

    for i in range(n):
        mask = (all_masks[i]['segmentation'].astype(np.uint8)) * 255
        is_paper, metrics = is_paper_like(mask, img.shape[:2])

        # Top row: Show mask
        axes[0, i].imshow(mask, cmap='gray')
        title = f"Object {i+1}\n"
        title += "✓ PAPER" if is_paper else f"✗ {metrics['reason']}"
        axes[0, i].set_title(title, color='green' if is_paper else 'red',
                            fontweight='bold')
        axes[0, i].axis('off')

        # Bottom row: Show overlay on image
        overlay = img.copy()
        colored_mask = np.zeros_like(img)
        colored_mask[mask > 0] = [0, 255, 0] if is_paper else [255, 0, 0]
        overlay = cv2.addWeighted(overlay, 0.7, colored_mask, 0.3, 0)

        axes[1, i].imshow(overlay)

        # Show metrics
        info = ""
        if is_paper:
            info = f"Score: {metrics['score']:.2f}\n"
            info += f"Area: {metrics['area_ratio']:.1%}\n"
            info += f"Aspect: {metrics['aspect_ratio']:.1f}\n"
            info += f"Rect: {metrics['rectangularity']:.0%}"
        else:
            if 'area_ratio' in metrics:
                info = f"Area: {metrics.get('area_ratio', 0):.1%}"
            if 'aspect_ratio' in metrics:
                info += f"\nAspect: {metrics.get('aspect_ratio', 0):.1f}"

        axes[1, i].set_title(info, fontsize=9)
        axes[1, i].axis('off')

    plt.tight_layout()
    plt.show()


# =============================================================================
# SAM PAPER DETECTOR CLASS
# =============================================================================

class SAM_PaperDetector:
    """
    SAM-based paper detector with geometric filtering
    Uses Meta's Segment Anything Model for robust paper segmentation
    """

    def __init__(self, model_type="vit_b"):
        """
        Initialize SAM model

        Args:
            model_type: 'vit_b' (375MB), 'vit_l' (1.2GB), or 'vit_h' (2.4GB)
                       vit_b is recommended - fast and accurate enough
        """
        print(f"🔄 Initializing SAM model ({model_type})...")

        self.model_type = model_type
        self.device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
        print(f"   Device: {self.device}")

        # Download checkpoint if not exists
        checkpoint_path = self._download_checkpoint(model_type)

        # Load SAM model
        print("📂 Loading SAM model...")
        self.sam = sam_model_registry[model_type](checkpoint=checkpoint_path)
        self.sam.to(device=self.device)

        # Create mask generator for automatic segmentation
        self.mask_generator = SamAutomaticMaskGenerator(
            model=self.sam,
            points_per_side=32,
            pred_iou_thresh=0.88,
            stability_score_thresh=0.95,
            crop_n_layers=1,
            crop_n_points_downscale_factor=2,
            min_mask_region_area=1000,
        )

        print("✓ SAM model ready!")

    def _download_checkpoint(self, model_type):
        """Download SAM checkpoint if not exists"""
        checkpoint_urls = {
            'vit_b': 'https://dl.fbaipublicfiles.com/segment_anything/sam_vit_b_01ec64.pth',
            'vit_l': 'https://dl.fbaipublicfiles.com/segment_anything/sam_vit_l_0b3195.pth',
            'vit_h': 'https://dl.fbaipublicfiles.com/segment_anything/sam_vit_h_4b8939.pth',
        }

        cache_dir = Path.home() / ".cache" / "sam_models"
        cache_dir.mkdir(parents=True, exist_ok=True)

        checkpoint_path = cache_dir / f"sam_{model_type}.pth"

        if not checkpoint_path.exists():
            print(f"📥 Downloading SAM {model_type} checkpoint...")
            print(f"   URL: {checkpoint_urls[model_type]}")
            print(f"   This may take a few minutes...")

            urllib.request.urlretrieve(
                checkpoint_urls[model_type],
                checkpoint_path,
                reporthook=self._download_progress
            )
            print(f"\n✓ Downloaded to {checkpoint_path}")
        else:
            print(f"✓ Using cached checkpoint: {checkpoint_path}")

        return str(checkpoint_path)

    def _download_progress(self, block_num, block_size, total_size):
        """Show download progress"""
        downloaded = block_num * block_size
        percent = min(downloaded / total_size * 100, 100)
        print(f"\r   Progress: {percent:.1f}%", end='', flush=True)

    def detect_paper_smart(self, img: np.ndarray, verbose: bool = True) -> Tuple[np.ndarray, List[Dict]]:
        """
        Detect paper using SAM + geometric filtering
        This is the NEW improved method that replaces detect_paper_automatic

        Args:
            img: RGB image (numpy array)
            verbose: Print diagnostic info

        Returns:
            (mask, all_masks) - best paper mask and all detected masks
        """
        if verbose:
            print("🔍 Running SAM automatic segmentation...")

        # Generate all masks in the image
        all_masks = self.mask_generator.generate(img)

        if len(all_masks) == 0:
            raise RuntimeError("❌ No objects detected by SAM")

        if verbose:
            print(f"   Found {len(all_masks)} objects")

        # Find best paper using geometric filtering
        best_mask, metrics = find_best_paper_mask(
            all_masks,
            img_shape=img.shape[:2],
            verbose=verbose
        )

        if best_mask is None:
            raise RuntimeError("❌ No paper-like object detected. Check image or adjust thresholds.")

        return best_mask, all_masks

    def detect_paper_with_box(self, img, box):
        """
        Detect paper using bounding box prompt (unchanged from original)
        """
        print("🔍 Running SAM with box prompt...")

        predictor = SamPredictor(self.sam)
        predictor.set_image(img)

        masks, scores, logits = predictor.predict(
            box=np.array(box),
            multimask_output=False
        )

        mask = (masks[0] * 255).astype(np.uint8)

        print(f"✓ Paper detected with box prompt (score: {scores[0]:.3f})")

        return mask


# =============================================================================
# PAPER SEGMENTATION FROM MASK
# =============================================================================

def segment_paper_from_mask(img, mask, debug=False):
    """
    Extract paper region using SAM mask (unchanged from original)
    """
    # Clean up mask
    kernel = np.ones((5, 5), np.uint8)
    mask = cv2.morphologyEx(mask, cv2.MORPH_CLOSE, kernel, iterations=2)
    mask = cv2.morphologyEx(mask, cv2.MORPH_OPEN, kernel, iterations=1)

    # Find paper contour
    contours, _ = cv2.findContours(mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)

    if not contours:
        raise RuntimeError("❌ No contours found in SAM mask")

    largest_contour = max(contours, key=cv2.contourArea)

    # Create clean mask from contour
    clean_mask = np.zeros_like(mask)
    cv2.drawContours(clean_mask, [largest_contour], -1, 255, -1)

    # Get bounding rectangle with padding
    x, y, w, h = cv2.boundingRect(largest_contour)
    pad_x = int(w * 0.05)
    pad_y = int(h * 0.05)
    x = max(0, x - pad_x)
    y = max(0, y - pad_y)
    w = min(img.shape[1] - x, w + 2 * pad_x)
    h = min(img.shape[0] - y, h + 2 * pad_y)

    # Crop image and mask
    cropped_img = img[y:y + h, x:x + w].copy()
    cropped_mask = clean_mask[y:y + h, x:x + w].copy()

    # Apply mask with white background
    cropped_mask = cropped_mask.astype(np.uint8)
    _, cropped_mask = cv2.threshold(cropped_mask, 127, 255, cv2.THRESH_BINARY)

    # Check if mask needs inversion
    white_pixels = np.sum(cropped_mask == 255)
    black_pixels = np.sum(cropped_mask == 0)

    if black_pixels > white_pixels:
        print("   ⚠️  Mask inverted, fixing...")
        cropped_mask = 255 - cropped_mask

    # Create white background and apply mask
    result = np.full_like(cropped_img, 255, dtype=np.uint8)
    paper_pixels = cropped_mask == 255
    result[paper_pixels] = cropped_img[paper_pixels]

    if debug:
        plt.figure(figsize=(20, 5))
        plt.subplot(151)
        plt.imshow(img)
        plt.title("1. Original Image")
        plt.axis('off')
        plt.subplot(152)
        plt.imshow(mask, cmap='gray')
        plt.title(f"2. SAM Mask")
        plt.axis('off')
        plt.subplot(153)
        plt.imshow(cropped_img)
        plt.title("3. Cropped Image")
        plt.axis('off')
        plt.subplot(154)
        plt.imshow(cropped_mask, cmap='gray')
        plt.title(f"4. Cropped Mask")
        plt.axis('off')
        plt.subplot(155)
        plt.imshow(result)
        plt.title("5. Final Result")
        plt.axis('off')
        plt.tight_layout()
        plt.show()

    return result


# =============================================================================
# IMAGE PROCESSING FUNCTIONS
# =============================================================================

def normalize_orientation(img):
    """Rotate portrait to landscape"""
    h, w = img.shape[:2]
    if h > w:
        img = cv2.rotate(img, cv2.ROTATE_90_CLOCKWISE)
        print(f"   ↻ Rotated from portrait ({w}×{h}) to landscape ({h}×{w})")
    else:
        print(f"   → Already landscape ({w}×{h})")
    return img


def clean_white_background(img):
    """Make white areas pure white, preserve colored strokes"""
    lab = cv2.cvtColor(img, cv2.COLOR_RGB2LAB)
    l_channel, a_channel, b_channel = cv2.split(lab)

    bright_mask = l_channel > 200
    l_channel[bright_mask] = 255

    lab_cleaned = cv2.merge((l_channel, a_channel, b_channel))
    result = cv2.cvtColor(lab_cleaned, cv2.COLOR_LAB2RGB)

    return result


def enhance_crayon_colors(img):
    """Gently boost saturation to make crayons vivid"""
    hsv = cv2.cvtColor(img, cv2.COLOR_RGB2HSV)
    h_channel, s_channel, v_channel = cv2.split(hsv)

    s_channel = s_channel.astype(np.float32)
    s_channel = s_channel * 1.15 + 5
    s_channel = np.clip(s_channel, 0, 255).astype(np.uint8)

    v_channel = v_channel.astype(np.float32)
    v_channel = v_channel * 1.02
    v_channel = np.clip(v_channel, 0, 255).astype(np.uint8)

    hsv_enhanced = cv2.merge((h_channel, s_channel, v_channel))
    result = cv2.cvtColor(hsv_enhanced, cv2.COLOR_HSV2RGB)

    return result


def resize_to_dataset(img, target_width=512, target_height=362):
    """
    Resize image to exact dataset dimensions with padding
    """
    h, w = img.shape[:2]

    scale = min(target_width / w, target_height / h)
    new_width = int(w * scale)
    new_height = int(h * scale)

    resized = cv2.resize(img, (new_width, new_height), interpolation=cv2.INTER_AREA)

    canvas = np.ones((target_height, target_width, 3), dtype=np.uint8) * 255

    y_offset = (target_height - new_height) // 2
    x_offset = (target_width - new_width) // 2

    canvas[y_offset:y_offset + new_height, x_offset:x_offset + new_width] = resized

    print(f"   📐 Resized: {w}×{h} → {new_width}×{new_height} → {target_width}×{target_height} (with padding)")

    return canvas


# =============================================================================
# COMPLETE PIPELINE
# =============================================================================

def full_pipeline_sam(image_path, detector, debug=False, save_path="output_sam.png",
                     show_analysis=False):
    """
    Complete SAM-based processing pipeline with geometric filtering

    Args:
        image_path: Path to input image
        detector: SAM_PaperDetector instance
        debug: Show intermediate steps
        save_path: Output file path
        show_analysis: Show detailed analysis of all detected objects

    Returns:
        Final processed image
    """

    print(f"\n{'=' * 70}")
    print(f"🎨 Processing: {image_path}")
    print(f"{'=' * 70}")

    # Load image
    img = cv2.imread(image_path)
    if img is None:
        raise RuntimeError(f"❌ Cannot read image: {image_path}")

    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    print(f"✓ Loaded image: {img.shape[1]}×{img.shape[0]} pixels")

    # Step 1: Detect Paper with SAM + Geometric Filtering
    print("\n🔍 Step 1: Detecting paper with SAM + geometric filtering...")
    mask, all_masks = detector.detect_paper_smart(img, verbose=True)

    # Optional: Show detailed analysis
    if show_analysis:
        print("\n📊 Showing detection analysis...")
        visualize_paper_detection_analysis(img, all_masks, max_display=6)

    # Step 2: Segment Paper
    print("\n✂️  Step 2: Segmenting paper from background...")
    paper = segment_paper_from_mask(img, mask, debug=debug)
    print(f"✓ Paper extracted: {paper.shape[1]}×{paper.shape[0]}")

    # Step 3: Normalize Orientation
    print("\n🔄 Step 3: Normalizing orientation...")
    paper = normalize_orientation(paper)

    # Step 4: Clean Background
    print("\n🧹 Step 4: Cleaning white background...")
    paper = clean_white_background(paper)
    print("✓ Background cleaned")

    # Step 5: Enhance Colors
    print("\n🎨 Step 5: Enhancing crayon colors...")
    paper = enhance_crayon_colors(paper)
    print("✓ Colors enhanced")

    # Step 6: Resize
    print("\n📏 Step 6: Resizing to dataset format...")
    final = resize_to_dataset(paper, target_width=512)

    # Save output
    print(f"\n💾 Saving output to: {save_path}")
    output_bgr = cv2.cvtColor(final, cv2.COLOR_RGB2BGR)
    cv2.imwrite(save_path, output_bgr)

    print(f"\n{'=' * 70}")
    print("✅ PIPELINE COMPLETE!")
    print(f"{'=' * 70}\n")

    return final


# =============================================================================
# BATCH PROCESSING
# =============================================================================

def process_batch_sam(image_folder, output_folder, detector):
    """
    Process multiple images with SAM
    """
    Path(output_folder).mkdir(parents=True, exist_ok=True)

    extensions = ['.jpg', '.jpeg', '.png', '.JPG', '.JPEG', '.PNG']
    image_files = []
    for ext in extensions:
        image_files.extend(Path(image_folder).glob(f'*{ext}'))

    print(f"\n📁 Found {len(image_files)} images to process")

    success_count = 0
    fail_count = 0

    for i, img_path in enumerate(image_files, 1):
        try:
            print(f"\n[{i}/{len(image_files)}] Processing: {img_path.name}")

            output_path = Path(output_folder) / f"sam_{img_path.name}"

            final = full_pipeline_sam(
                image_path=str(img_path),
                detector=detector,
                debug=False,
                save_path=str(output_path),
                show_analysis=False
            )

            success_count += 1

        except Exception as e:
            print(f"❌ Failed: {e}")
            fail_count += 1

    print(f"\n{'=' * 70}")
    print(f"✅ Successfully processed: {success_count}")
    print(f"❌ Failed: {fail_count}")
    print(f"{'=' * 70}")


# =============================================================================
# MAIN EXECUTION
# =============================================================================

if __name__ == "__main__":

    # Initialize SAM Detector (Do this ONCE)
    print("🚀 Initializing SAM Paper Detector...")
    print("   Using 'vit_b' model (375MB) - good balance of speed and accuracy")
    print("   For best accuracy, use 'vit_h' (2.4GB) instead")

    detector = SAM_PaperDetector(model_type='vit_b')

    # Process single image
    try:
        final_result = full_pipeline_sam(
            image_path="ml-models/image/data/wood_bg2.jpeg",
            detector=detector,
            debug=True,  # Show intermediate steps
            save_path="ml-models/image/data/output/final_sam_output.png",
            show_analysis=True  # Show detailed detection analysis
        )

        # Display result
        plt.figure(figsize=(10, 7))
        plt.imshow(final_result)
        plt.title("Final SAM Result with Geometric Filtering", fontsize=16, fontweight='bold')
        plt.axis('off')
        plt.tight_layout()
        plt.show()

    except Exception as e:
        print(f"\n❌ Error: {e}")
        import traceback
        traceback.print_exc()

    # Uncomment for batch processing:
    # process_batch_sam("input_images/", "output_images/", detector)